# Read Table -> Establish Generic Silver-level Table

In [0]:
#Imports to run API Data Scraping script

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datetime import datetime
import logging
import traceback
import json
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *


In [0]:
#parameters that need to be changed between notebooks

silver_level_schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('transaction_date', DateType(), True),
    StructField('transaction_timestamp', TimestampType (), True),
    StructField('client_id', StringType(), True),
    StructField('card_id', StringType(), True),
    StructField('amount', DecimalType(), True),
    StructField('use_chip', StringType(), True),
    StructField('merchant_id', IntegerType(), True),
    StructField('merchant_city', StringType(), True),
    StructField('merchant_state', StringType(), True),
    StructField('zip', IntegerType(), True),
    StructField('mcc', IntegerType(), True),
    StructField('errors', StringType(), True)
])

schema = silver_level_schema
schema_path = 'financial_transactions_dataset.default'
volume_path = '/Volumes/financial_transactions_dataset/default/financial_transactions_dataset_analytics_volume'
volume_identifier = 'transactions_data'
silver_table_name = 'silver_transactions_data'
bronze_table_name = 'bronze_transactions_data'

In [0]:
#Set up logger to track pipeline activities

def setup_logging():

    # Ensure logs directory exists before starting logging
    os.makedirs('logs', exist_ok=True)

    logging.basicConfig(
        level = logging.INFO,
        format = '%(asctime)s - %(levelname)s - %(message)s',
        handlers = [logging.FileHandler(f'logs/silver_level_table_run_{datetime.now().strftime('%Y%m%d')}.log'),
                    logging.StreamHandler(sys.stdout)
                   ]
    )

    return logging.getLogger(__name__)

In [0]:
def dollar_fields_cleaned(logger, df):
    logger.info(f'Converting amount to decimal')

    df = df.withColumn('amount', F.regexp_replace(F.col('amount'), '\\$', ''))
    df = df.withColumn('amount', F.col('amount').cast('decimal(18,2)'))

    return df

In [0]:
def split_date_column(logger, df):
    logger.info(f'Splitting date and time for each record.')

    df = df.withColumn('transaction_date', F.split(F.col('date'), ' ').getItem(0))
    df = df.withColumn('transaction_timestamp', F.split(F.col('date'), ' ').getItem(1))

    logger.info(f'Dropping original date field.')

    df = df.drop('date')

    return df

In [0]:
def convert_fields_to_int(logger, df):
    logger.info(f'Converting fields to integer')

    df = df.withColumn('id', F.col('id').cast('int'))
    df = df.withColumn('client_id', F.col('client_id').cast('int'))
    df = df.withColumn('merchant_id', F.col('merchant_id').cast('int'))
    df = df.withColumn('zip', F.col('zip').cast('decimal(18,0)').cast('int'))
    df = df.withColumn('mcc', F.col('mcc').cast('decimal(18,0)').cast('int'))

    return df

In [0]:
def rename_columns(logger, df):
    logger.info(f'Renaming columns')
    
    df = df.withColumnsRenamed({"zip": "zip_code", "mcc": "mcc_code"})

    return df 

In [0]:
def reorder_fields(logger, df):
    logger.info(f'Reordering arrangement of fields')

    df = df.select(
        'id',
        'transaction_date',
        'transaction_timestamp',
        'client_id',
        'card_id',
        'amount',
        'use_chip',
        'merchant_id',
        'merchant_city',
        'merchant_state',
        'zip_code',
        'mcc_code',
        'errors'
    )

    return df

In [0]:
#Check if a table exists in a schema; create an empty table if it does not exist.

def check_or_create_table(logger, silver_table_name, schema_path):

    logger.info(f'Checking to see if {silver_table_name} exists.')
    
    if spark.catalog.tableExists(f'{schema_path}.{silver_table_name}'):
        logger.info("Table exists; table creation skipped.")
        flag = 1
    else:
        logger.info('Table does not exist; table will be created.')
        flag = 0
    
    return flag


In [0]:
#Pull data from bronze-level preliminary table

    #Function to read a SQL data table as a dataframe

def read_table_as_df(logger, schema_path, bronze_table_name):

    logger.info(f'Reading data from {schema_path}.{bronze_table_name}')

    df = spark.read.table(f'{schema_path}.{bronze_table_name}')
    total_row_count = df.count()

    logger.info(f'Read {total_row_count} rows from {schema_path}.{bronze_table_name}')
    
    return df


In [0]:
#Function to overwrite/append data to silver level table

def create_or_append_to_table(logger, table_exists, df, schema_path, silver_table_name):

    if table_exists == 1:
        logger.info(f'Appending data to {schema_path}')

        df.write.format('delta').mode('append').saveAsTable(f'{schema_path}.{silver_table_name}')

        total_row_count = df.count()

        logger.info(f'Wrote {total_row_count} rows to {schema_path}')
    
    else:

        logger.info(f'Creating table to {schema_path}')

        df.write.format('delta').mode('overwrite').saveAsTable(f'{schema_path}.{silver_table_name}')

        total_row_count = df.count()

        logger.info(f'Wrote {total_row_count} rows to {schema_path}')
    
    return


In [0]:
def silver_level_transformation(silver_level_schema, schema_path, bronze_table_name, silver_table_name, volume_path, volume_identifier):

    schema = silver_level_schema
    schema_path = schema_path
    volume_path = volume_path
    volume_identifier = volume_identifier
    silver_table_name = silver_table_name
    bronze_table_name = bronze_table_name
    
    logger = setup_logging()

    logger.info(f'Initializing ETL on bronze dataframe')

    is_created = check_or_create_table(logger, silver_table_name, schema_path)
    bronze_df = read_table_as_df(logger, schema_path, bronze_table_name)
    dollar_df = dollar_fields_cleaned(logger, bronze_df)
    split_df = split_date_column(logger, dollar_df)
    int_df = convert_fields_to_int(logger, split_df)
    rename_df = rename_columns(logger, int_df)
    silver_df = reorder_fields(logger, rename_df)

    create_or_append_to_table(logger, is_created, silver_df, schema_path, silver_table_name)
    
    logger.info(f'Silver level table {silver_table_name} created or appended to successfully.')

    return


In [0]:
silver_level_transformation(silver_level_schema, schema_path, bronze_table_name, silver_table_name, volume_path, volume_identifier)

In [0]:
%sql
SELECT * FROM financial_transactions_dataset.default.silver_transactions_data LIMIT 10